# Teaser: solución numérica de la ecuación de difusión

En esta notebook vas a **ejecutar** el mismo cálculo escrito de tres formas distintas, **cronometrar** cuánto tarda cada una, y al final **escribir tu propia conclusión** sobre por qué la diferencia es tan grande. No hace falta que entiendas cada línea de código a la perfección — lo importante es que observes el resultado y pienses por qué pasa.

## El problema: la difusión

Imagina que pones una gota de tinta en un vaso de agua quieta. Al principio está concentrada en un punto; con el tiempo se esparce hasta que el agua queda de un color uniforme. Eso es difusión: algo acumulado en un lugar se reparte hacia donde hay menos.

Para simularlo en la computadora:

* Dividimos el espacio en una **grilla** (como una hoja cuadriculada). Cada casilla de la grilla guarda un número: cuánta "tinta" (o calor, o concentración) hay ahí.
* En cada paso de tiempo, cada casilla se actualiza mirando solo a sus 4 vecinas (arriba, abajo, izquierda, derecha): si tiene más que el promedio de sus vecinas, pierde un poco; si tiene menos, gana un poco. Repetido muchas veces, esto reproduce el esparcimiento real.
* Las **condiciones de frontera periódicas** son un truco para que la grilla no tenga bordes raros: lo que "sale" por la derecha reaparece por la izquierda, como en Pac-Man.

Esto se escribe formalmente como una ecuación diferencial parcial (PDE):

$$\partial_t u = \alpha (\partial_x^2 + \partial_y^2) u$$

y se discretiza (se convierte en algo que la computadora puede calcular paso a paso) con esta fórmula, que actualiza el valor de un punto $(x_i, y_j)$ del paso de tiempo $t_n$ al siguiente $t_{n+1}$:

$$ u_{i,j}^{(n+1)} = u_{i,j}^{(n)} + \frac{\alpha \Delta t}{\Delta x^2} \left(
u_{i-1,j}^{(n)}
+u_{i+1,j}^{(n)}
+u_{i,j-1}^{(n)}
+u_{i,j+1}^{(n)}
-4u_{i,j}^{(n)}
\right) $$

No memorices la fórmula — quédate con la idea: **el nuevo valor de cada casilla depende del valor actual de sus 4 vecinas**. Con una grilla de 128×128 y 400 pasos de tiempo, esa fórmula se evalúa más de 6.5 millones de veces. Ahí es donde empieza a importar *cómo* escribes el código.

## Por qué vas a ver tres versiones del mismo código

Las tres versiones calculan exactamente lo mismo. Lo único que cambia es *cómo* está escrito en Python:

1. **Loops puros** — le decimos a Python, casilla por casilla, "actualízate a ti misma". Es como lavar platos uno por uno, a mano.
2. **NumPy vectorizado** — le decimos a Python "actualiza toda la grilla de un solo golpe". NumPy delega ese trabajo a código en C ya compilado y optimizado. Es como meter todos los platos juntos a un lavavajillas.
3. **Numba (JIT)** — Numba traduce tu función de Python a código máquina la primera vez que se ejecuta (JIT = *Just-In-Time*, "justo a tiempo"). Por eso, antes de medir el tiempo, hacemos una llamada de "calentamiento": la primera ejecución incluye el tiempo de traducción, y no queremos que eso se mezcle con el tiempo real de cálculo.

Vas a usar `%%time` al inicio de cada celda para medir cuánto tarda en ejecutarse. Al final de la salida vas a ver una línea como:

```
Wall time: 4.21 s
```

Ese número (`Wall time`, tiempo real transcurrido) es el que vas a anotar para comparar las tres versiones.

## Preparar la simulación

Primero desactivamos el paralelismo interno de NumPy (OpenMP), para que las tres versiones corran en un solo hilo de CPU y la comparación de tiempos sea justa.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_NUM_THREADS"] = "1"


Ahora definimos los parámetros de la simulación y el estado inicial de la grilla:

* `n_points`: tamaño de la grilla (128 x 128 casillas).
* `n_iterations`: cuántos pasos de tiempo vamos a simular.
* `dt`: qué tan grande es cada paso de tiempo.
* `D`: qué tan rápido se difunde (el $\alpha$ de la fórmula).

**Actividad:** ejecuta la siguiente celda. Vas a ver un mapa de calor con la condición inicial: una zona con más "tinta" (colores más claros) rodeada de zonas sin nada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# parametros de la simulacion
n_iterations = 400   # cuantos pasos de tiempo simulamos
n_points = 128        # tamano de la grilla (n_points x n_points)
dt = 0.1              # paso de tiempo
D = 1.0               # coeficiente de difusion (alpha en la formula)

def init(val=0.005):
    x = np.linspace(0., 4.*np.pi, num=n_points+2)
    y = np.linspace(0., 4.*np.pi, num=n_points+2)
    grid = val * np.outer(np.sin(x)**4, np.sin(y)**4)
    return grid

grid = init()
plt.pcolormesh(grid.T); plt.axis('image'); plt.show()

## Actividad 1: versión Python puro con loops

Este es el código más directo: un `for` por cada casilla de la grilla, por cada paso de tiempo. Es fácil de leer y de relacionar con la fórmula matemática — pero, como vas a comprobar, es la opción más lenta.

**Instrucciones:** ejecuta la celda y espera a que termine (puede tardar varios segundos). Al final de la salida busca la línea `Wall time: ...` y anótala.

In [ ]:
%%time
grid = init()
grid_tmp = np.zeros_like(grid)
for k in range(1, n_iterations+1):
    for j in range(n_points + 2):      # condiciones de frontera periodicas
        grid[ 0, j] = grid[-2, j]
        grid[-1, j] = grid[ 1, j]
    for i in range(n_points + 2):
        grid[ i,-1] = grid[ i, 1]
        grid[ i, 0] = grid[ i,-2]
    for i in range(1, n_points+1):     # formula de diferencias finitas
        for j in range(1, n_points+1):
            grid_tmp[i, j] = grid[i, j] + dt * D * (
                grid[i-1, j] + grid[i+1, j] + grid[i, j-1] + grid[i, j+1] - 4.0 * grid[i, j])
    for i in range(1, n_points+1):     # actualizar la grilla
        for j in range(1, n_points+1):
            grid[i, j] = grid_tmp[i, j]
plt.pcolormesh(grid.T); plt.axis('image'); plt.show()

**Tus resultados — Versión Python con loops**

Haz doble clic en esta celda y reemplaza la línea de abajo con el `Wall time` que obtuviste:

- Wall time: ___ s

## Actividad 2: versión NumPy (vectorizada)

Ahora, en vez de recorrer la grilla casilla por casilla, le pedimos a NumPy que opere sobre **toda la grilla a la vez**, usando *slicing* (rebanado) de arreglos: `grid[1:-1, 1:-1]` selecciona de un solo golpe todas las casillas "interiores" de la grilla. Esto se llama **vectorización**.

**Instrucciones:** ejecuta las dos celdas siguientes y anota de nuevo el `Wall time`.

In [ ]:
grid = init()

In [ ]:
%%time
for k in range(1, n_iterations+1):
    grid[ 0, :] = grid[-2, :]    # condiciones de frontera periodicas
    grid[-1, :] = grid[ 1, :]
    grid[ :,-1] = grid[ :, 1]
    grid[ :, 0] = grid[ :,-2]
    # formula de diferencias finitas aplicada a arreglos completos
    grid[1:-1, 1:-1] = grid[1:-1, 1:-1] + dt * D * (
        grid[0:-2, 1:-1] + grid[2:  , 1:-1] + grid[1:-1, 0:-2] + grid[1:-1, 2:  ] - 4.0 * grid[1:-1, 1:-1])

#plt.pcolormesh(grid.T); plt.axis('image'); plt.show()

**Tus resultados — Versión NumPy**

- Wall time: ___ s
- ¿Cuántas veces más rápida fue esta versión comparada con la de loops? (tiempo de loops ÷ tiempo de NumPy): ___

## Actividad 3: compilación Just-In-Time con Numba

**Numba** puede compilar una función de Python a código máquina la primera vez que se ejecuta. Fíjate que el código de la función `evolve_numba` de abajo es casi idéntico al de la versión NumPy — lo único que agregamos es el decorador `@numba.jit` encima.

In [ ]:
import numba

In [ ]:
@numba.jit
def evolve_numba(grid, dt, D, n_iterations):
    for k in range(1, n_iterations+1):
        grid[ 0, :] = grid[-2, :]    # condiciones de frontera periodicas
        grid[-1, :] = grid[ 1, :]
        grid[ :,-1] = grid[ :, 1]
        grid[ :, 0] = grid[ :,-2]
        # formula de diferencias finitas aplicada a arreglos completos
        grid[1:-1, 1:-1] = grid[1:-1, 1:-1] + dt * D * (
            grid[0:-2, 1:-1] + grid[2:  , 1:-1] + grid[1:-1, 0:-2] + grid[1:-1, 2:  ] - 4.0 * grid[1:-1, 1:-1])

Antes de medir el tiempo, necesitamos una llamada de "calentamiento" (*warmup*): la primera vez que Numba ve esta función tiene que traducirla a código máquina, y eso toma tiempo. Si midiéramos esa primera llamada, estaríamos midiendo "tiempo de traducción + tiempo de cálculo" mezclados. Por eso la llamamos una vez con pocas iteraciones (10) solo para forzar la compilación, sin medir el tiempo.

**Instrucciones:** ejecuta las tres celdas siguientes en orden.

In [ ]:
grid = init()
# llamada de calentamiento para forzar la compilacion JIT (no medimos el tiempo)
evolve_numba(grid, dt, D, 10)

In [ ]:
grid = init()

In [ ]:
%%time
evolve_numba(grid, dt, D, n_iterations)
#plt.pcolormesh(grid.T); plt.axis('image'); plt.show()

**Tus resultados — Versión Numba (JIT)**

- Wall time: ___ s
- ¿Cuántas veces más rápida fue esta versión comparada con la de loops? ___
- ¿Fue más rápida, más lenta, o parecida a la versión NumPy? ___

## ¿Por qué pasa esto?

* **Python puro** se *interpreta* línea por línea, no se compila de antemano. La mayoría de las optimizaciones que hace un compilador (como para C o Fortran) no son posibles, así que el resultado es mucho más lento.
* **NumPy** hace el cómputo sobre arreglos usando código ya compilado en C y Fortran, altamente optimizado. Python solo llama a esas rutinas desde una capa de alto nivel — el trabajo pesado no lo hace Python.
* **Numba** compila tu propio código Python (loops, expresiones con NumPy) a código máquina la primera vez que se ejecuta. El rendimiento resultante puede ser comparable al de C o Fortran, sin que tengas que escribir una sola línea de C.

## Actividad final: escribe tu conclusión

Edita esta celda (doble clic) y responde en 4-6 líneas, usando tus propios resultados:

1. ¿Cuál de las tres versiones fue más rápida en tu computadora?
2. ¿La diferencia entre la versión de loops y la de NumPy fue grande o pequeña? ¿Te sorprendió?
3. En tus palabras: ¿por qué escribir el "mismo" cálculo de forma distinta puede cambiar tanto el tiempo de ejecución?
4. Ahora prueba cambiar `n_points` de 128 a 256 en la celda de parámetros y vuelve a correr las tres versiones. ¿El tiempo de cuál versión creció más? ¿Por qué crees que pasó eso?

**Tu conclusión:**

_(escribe aquí)_